# Mountain NER — Demo

This notebook demonstrates inference with the fine-tuned `bert-base-cased` model.

**Prerequisites**
- Install dependencies: `pip install -r requirements.txt`
- Either run `python train.py` first (saves weights to `model_output/`),
  or set `MODEL_PATH` below to a HuggingFace Hub model ID.


In [1]:
from inference import build_ner_pipeline, extract_mountains
from IPython.display import display, HTML
import json

MODEL_PATH = "model_output"

ner = build_ner_pipeline(MODEL_PATH)
print("Model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded.


## 1. Single sentence examples

In [2]:
DEMO_SENTENCES = [
    "Last summer, our team attempted to summit Mount Everest but turned back due to weather.",
    "The documentary features stunning aerial footage of K2 and Kangchenjunga.",
    "Mont Blanc stands at 4808 meters above sea level.",
    "Kilimanjaro attracts thousands of mountaineers every climbing season.",
    "Unlike Denali, Aconcagua requires no technical ice climbing.",
    "The Amazon River runs through South America.",
    "Scientists are studying environmental changes around Lake Baikal.",
    "She bought a new laptop for her studies.",
    "Our journey took us past the Matterhorn and through the valleys below Mont Blanc.",
    "Mount Sinai is mentioned in historical texts alongside Mount Ararat.",
]

for sent in DEMO_SENTENCES:
    mountains = extract_mountains(sent, ner)
    found = [m['word'] for m in mountains] if mountains else []
    label = '\u2705' if found else '\u25cb'
    print(f"{label} {sent}")
    if found:
        for m in mountains:
            print(f"     \u2514 {m['word']!r:30s} score={m['score']:.3f}")
    print()

✅ Last summer, our team attempted to summit Mount Everest but turned back due to weather.
     └ 'Mount Everest'                score=0.994



✅ The documentary features stunning aerial footage of K2 and Kangchenjunga.
     └ 'K2'                           score=0.693
     └ 'Kangchenjunga'                score=0.653

✅ Mont Blanc stands at 4808 meters above sea level.
     └ 'Mont Blanc'                   score=0.986



✅ Kilimanjaro attracts thousands of mountaineers every climbing season.
     └ 'Kilimanjaro'                  score=0.529



✅ Unlike Denali, Aconcagua requires no technical ice climbing.
     └ 'Denali'                       score=0.778
     └ 'Aconcagua'                    score=0.889

○ The Amazon River runs through South America.

○ Scientists are studying environmental changes around Lake Baikal.



○ She bought a new laptop for her studies.



✅ Our journey took us past the Matterhorn and through the valleys below Mont Blanc.
     └ 'Matterhorn'                   score=0.989
     └ 'Mont Blanc'                   score=0.986

✅ Mount Sinai is mentioned in historical texts alongside Mount Ararat.
     └ 'Mount Sinai'                  score=0.995
     └ 'Mount Ararat'                 score=0.953



## 2. Inline HTML highlighting

In [3]:
def highlight_html(text: str, mountains: list[dict]) -> str:
    if not mountains:
        return text
    result = []
    prev = 0
    for m in sorted(mountains, key=lambda x: x["start"]):
        result.append(text[prev:m["start"]])
        score = m["score"]
        mark_open = (
            '<mark style="background:#ffe680;border-radius:3px;padding:1px 3px;" '
            f'title="score={score}">'
        )
        result.append(mark_open + text[m["start"]:m["end"]] + "</mark>")
        prev = m["end"]
    result.append(text[prev:])
    return "".join(result)

PARA = (
    "Every mountaineer dreams of standing atop Mount Everest, but peaks like K2, "
    "Nanga Parbat, and Annapurna are equally formidable challenges. "
    "In Europe, Mont Blanc and the Matterhorn draw thousands of climbers each year. "
    "Meanwhile, Kilimanjaro offers a more accessible high-altitude experience in Africa."
)

mountains = extract_mountains(PARA, ner)
display(HTML(highlight_html(PARA, mountains)))
print(f"\nDetected {len(mountains)} mountain(s):")
for m in mountains:
    print(f"  {m['word']!r} (score={m['score']:.3f})")


Detected 7 mountain(s):
  'Mount Everest' (score=0.991)
  'K2' (score=0.820)
  'Nanga Parbat' (score=0.664)
  'Annapurna' (score=0.621)
  'Mont Blanc' (score=0.989)
  'Matterhorn' (score=0.990)
  'Kilimanjaro' (score=0.570)


## 3. Evaluation on the held-out test set

In [4]:
import json
from seqeval.metrics import classification_report

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

test_data = load_jsonl("dataset/test.jsonl")

true_seqs, pred_seqs = [], []

for ex in test_data:
    text = ex["text"]
    true_tags = ex["tags"]

    raw = ner(text)

    pred_tags = ["O"] * len(ex["tokens"])
    char_offset = 0
    token_starts = []
    remaining = text
    for tok in ex["tokens"]:
        idx = remaining.find(tok)
        token_starts.append(char_offset + idx)
        char_offset += idx + len(tok)
        remaining = remaining[idx + len(tok):]

    for span in raw:
        if span["entity_group"] != "MOUNTAIN":
            continue
        for ti, ts in enumerate(token_starts):
            if span["start"] <= ts < span["end"]:
                pred_tags[ti] = "B-MOUNTAIN" if ts == span["start"] else "I-MOUNTAIN"

    true_seqs.append(true_tags)
    pred_seqs.append(pred_tags)

print(classification_report(true_seqs, pred_seqs))

              precision    recall  f1-score   support

    MOUNTAIN       1.00      1.00      1.00        51

   micro avg       1.00      1.00      1.00        51
   macro avg       1.00      1.00      1.00        51
weighted avg       1.00      1.00      1.00        51



## 4. Error analysis — false positives & false negatives

In [5]:
false_positives, false_negatives = [], []

for ex in test_data:
    text = ex["text"]
    has_mountain = any(t != "O" for t in ex["tags"])
    detected = extract_mountains(text, ner)

    if detected and not has_mountain:
        false_positives.append((text, [m['word'] for m in detected]))
    elif not detected and has_mountain:
        true_mountain = next(ex["tokens"][i]
                             for i, t in enumerate(ex["tags"]) if t == "B-MOUNTAIN")
        false_negatives.append((text, true_mountain))

print(f"False positives: {len(false_positives)}")
for text, words in false_positives[:5]:
    print(f"  '{text}'  → detected {words}")

print(f"\nFalse negatives: {len(false_negatives)}")
for text, mt in false_negatives[:5]:
    print(f"  '{text}'  → missed '{mt}'")

False positives: 0

False negatives: 0
